# CIFAR-100 SHViT — DyT+Softmax — Experiment 1

Train the unmodified family baseline from scratch.

- Actual trained operators: **DyT + Softmax**.
- Fixed seed **42**, logical CUDA device **1** (`cuda:1`); no CPU/GPU-0 fallback.
- Original CIFAR split: 45,000 train / 5,000 validation / 10,000 test; original augmentation.
- Original SHViT: 4x4 patches, dim=128, depth=3, MLP ratio=4, dropout=0.1.
- Original training: 400 epochs, AdamW lr=0.001; weight decay=0.05; cosine LR; gradient clip=1.0.
- All retained and new parameters are trainable; no backbone freezing.
- Run notebook **1 in this family first** before 5/6/7. Keep the same working directory for all notebooks.
- Matching baseline: `./runs/dyt_softmax/1_cifar100_shvit_dyt_softmax_baseline_seed42/best.pt`.
- Results: `./runs/dyt_softmax/1_cifar100_shvit_dyt_softmax_baseline_seed42/`; validation selects `best.pt`, then test is evaluated.

Baseline attention is standard `Softmax(QK^T / sqrt(d), dim=-1)` over keys, with no Q/K L1 normalization. When installing LENSMax, only Softmax is replaced: LENSMax receives the same scaled QK scores. LENSNorm and LENSMax implementations are unchanged from the SimA experiments. Each replacement experiment independently loads this family's new Softmax baseline `best.pt`; all student parameters remain trainable, using classification cross-entropy without KD.

RMSNorm: per-token channel RMS, learned channel scale, no bias, eps=1e-6. DyT: learned scalar alpha initialized to 0.5, channel scale initialized to 1, channel bias initialized to 0.

References: [DyT CVPR 2025 paper](https://openaccess.thecvf.com/content/CVPR2025/papers/Zhu_Transformers_without_Normalization_CVPR_2025_paper.pdf), [RMSNorm definition](https://docs.pytorch.org/docs/stable/generated/torch.nn.RMSNorm.html).


In [1]:
import gc
import json
import math
import random
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms


EXPERIMENT_NAME = "1_cifar100_shvit_dyt_softmax_baseline"
FROM_PRETRAINED = False
USE_LENS_NORM = False
USE_LENS_MAX = False

SEEDS = [42]
DATA_ROOT = Path("./data")
BASELINE_FAMILY = "dyt_softmax"
BASELINE_NORM = "dyt"
BASELINE_EXPERIMENT_NAME = "1_cifar100_shvit_dyt_softmax_baseline"
RUNS_ROOT = Path("./runs") / BASELINE_FAMILY
RMS_NORM_EPS = 1e-6
DYT_ALPHA_INIT = 0.5

IMAGE_SIZE = 32
PATCH_SIZE = 4
NUM_CLASSES = 100
EMBED_DIM = 128
DEPTH = 3
MLP_RATIO = 4.0
DROPOUT = 0.1
ATTENTION_DROPOUT = 0.0

LENS_NORM_HIDDEN = 32
LENS_MAX_ROW_HIDDEN = 32
LENS_MAX_ELEMENT_HIDDEN = 16

BATCH_SIZE = 256
NUM_WORKERS = 4
VALIDATION_SIZE = 5000
EPOCHS = 400
LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.05
GRAD_CLIP_NORM = 1.0

# Use logical CUDA device 1; fail clearly instead of falling back to GPU 0/CPU.
GPU_INDEX = 1
if not torch.cuda.is_available() or torch.cuda.device_count() <= GPU_INDEX:
    raise RuntimeError("GPU 1 (cuda:1) is required. Expose at least two CUDA devices to this kernel.")
DEVICE = torch.device(f"cuda:{GPU_INDEX}")
torch.cuda.set_device(DEVICE)
print(f"device={DEVICE} | experiment={EXPERIMENT_NAME}")


/home/yuhao/anaconda3/envs/brevitas/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device=cuda:1 | experiment=1_cifar100_shvit_dyt_softmax_baseline


In [2]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


class RMSNorm(nn.Module):
    """Per-token RMS normalization with learned channel scale and no bias."""

    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        # Accumulate low-precision statistics in float32; preserve float64.
        stats = x.float() if x.dtype in (torch.float16, torch.bfloat16) else x
        inverse_rms = torch.rsqrt(stats.square().mean(dim=-1, keepdim=True) + self.eps)
        normalized = (stats * inverse_rms).to(x.dtype)
        return normalized * self.weight


class DynamicTanh(nn.Module):
    """DyT(x) = gamma * tanh(alpha * x) + beta; alpha starts at 0.5."""

    def __init__(self, dim, alpha_init=0.5):
        super().__init__()
        self.alpha = nn.Parameter(torch.tensor(float(alpha_init)))
        self.weight = nn.Parameter(torch.ones(dim))
        self.bias = nn.Parameter(torch.zeros(dim))

    def forward(self, x):
        return self.weight * torch.tanh(self.alpha * x) + self.bias


def make_baseline_norm(dim):
    if BASELINE_NORM == "rmsnorm":
        return RMSNorm(dim, eps=RMS_NORM_EPS)
    if BASELINE_NORM == "dyt":
        return DynamicTanh(dim, alpha_init=DYT_ALPHA_INIT)
    raise ValueError(f"Unknown baseline norm: {BASELINE_NORM}")


class LENSNorm(nn.Module):
    """Predicts one affine scale and bias for every token."""

    def __init__(self, dim, hidden_dim=32, scale_min=0.05, scale_max=12.0):
        super().__init__()
        self.scale_min = scale_min
        self.scale_max = scale_max
        self.predictor_0 = nn.Linear(dim, hidden_dim)
        self.predictor_1 = nn.Linear(hidden_dim, 2)
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.kaiming_normal_(self.predictor_0.weight, nonlinearity="relu")
        nn.init.zeros_(self.predictor_0.bias)
        nn.init.normal_(self.predictor_1.weight, mean=0.0, std=1e-3)
        with torch.no_grad():
            self.predictor_1.bias.copy_(torch.tensor([1.0, 0.0]))

    def forward(self, x):
        affine = self.predictor_1(F.relu(self.predictor_0(x)))
        scale = affine[..., 0:1].clamp(self.scale_min, self.scale_max)
        bias = affine[..., 1:2]
        return scale * x + bias


class LENSMax(nn.Module):
    """Maps each complete attention-score row to positive attention weights."""

    def __init__(self, num_tokens, row_hidden=32, element_hidden=16):
        super().__init__()
        self.num_tokens = num_tokens
        self.row_mlp = nn.Sequential(
            nn.Linear(num_tokens, row_hidden),
            nn.ReLU(),
            nn.Linear(row_hidden, 1),
        )
        self.element_mlp = nn.Sequential(
            nn.Linear(2, element_hidden),
            nn.ReLU(),
            nn.Linear(element_hidden, 1),
        )
        self.reset_parameters()

    def reset_parameters(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.kaiming_normal_(module.weight, nonlinearity="relu")
                nn.init.zeros_(module.bias)

    def forward(self, scores):
        batch_size, num_queries, num_keys = scores.shape
        if num_keys != self.num_tokens:
            raise ValueError(f"LENSMax expects {self.num_tokens} keys, received {num_keys}")

        flat_rows = scores.reshape(batch_size * num_queries, num_keys)
        row_state = self.row_mlp(flat_rows)
        row_state = row_state.unsqueeze(1).expand(-1, num_keys, -1)
        element_input = torch.cat((flat_rows.unsqueeze(-1), row_state), dim=-1)
        logits = self.element_mlp(element_input).squeeze(-1)

        divisor = float(2 ** int(math.floor(math.log2(num_keys))))
        weights = torch.sigmoid(logits) / divisor
        return weights.reshape(batch_size, num_queries, num_keys)


In [3]:
class SingleHeadSelfAttention(nn.Module):
    def __init__(self, dim, num_tokens, qkv_bias=True, attn_drop=0.0, proj_drop=0.0):
        super().__init__()
        self.scale = dim ** -0.5
        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)
        self.lens_max = None

    def forward(self, x):
        batch_size, num_tokens, dim = x.shape
        qkv = self.qkv(x).reshape(batch_size, num_tokens, 3, dim).permute(2, 0, 1, 3)
        q, k, v = qkv.unbind(0)
        scores = (q @ k.transpose(-2, -1)) * self.scale
        if self.lens_max is None:
            # Standard scaled dot-product attention, normalized over keys.
            attention = F.softmax(scores, dim=-1)
        else:
            # Replace only Softmax; LENSMax receives the same scaled QK scores.
            attention = self.lens_max(scores)

        attention = self.attn_drop(attention)
        x = attention @ v
        x = self.proj(x)
        return self.proj_drop(x)


class FeedForward(nn.Module):
    def __init__(self, dim, hidden_dim, drop=0.0):
        super().__init__()
        self.fc1 = nn.Linear(dim, hidden_dim)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_dim, dim)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        x = self.drop(self.act(self.fc1(x)))
        return self.drop(self.fc2(x))


class TransformerBlock(nn.Module):
    def __init__(self, dim, num_tokens, mlp_ratio=4.0, drop=0.0, attn_drop=0.0):
        super().__init__()
        self.norm1 = make_baseline_norm(dim)
        self.attn = SingleHeadSelfAttention(
            dim=dim,
            num_tokens=num_tokens,
            attn_drop=attn_drop,
            proj_drop=drop,
        )
        self.norm2 = make_baseline_norm(dim)
        self.mlp = FeedForward(dim, int(dim * mlp_ratio), drop=drop)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


class CIFAR100SHViT(nn.Module):
    def __init__(
        self,
        image_size=32,
        patch_size=4,
        num_classes=10,
        dim=128,
        depth=3,
        mlp_ratio=4.0,
        drop=0.1,
        attn_drop=0.0,
    ):
        super().__init__()
        self.dim = dim
        self.num_patches = (image_size // patch_size) ** 2
        self.num_tokens = self.num_patches + 1
        self.patch_embed = nn.Conv2d(3, dim, kernel_size=patch_size, stride=patch_size)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_tokens, dim))
        self.pos_drop = nn.Dropout(drop)
        self.blocks = nn.ModuleList([
            TransformerBlock(dim, self.num_tokens, mlp_ratio, drop, attn_drop)
            for _ in range(depth)
        ])
        self.norm = make_baseline_norm(dim)
        self.head = nn.Linear(dim, num_classes)
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(module):
        if isinstance(module, nn.Linear):
            nn.init.trunc_normal_(module.weight, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Conv2d):
            nn.init.kaiming_normal_(module.weight, mode="fan_out")
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.LayerNorm):
            nn.init.ones_(module.weight)
            nn.init.zeros_(module.bias)

    def forward(self, x):
        x = self.patch_embed(x).flatten(2).transpose(1, 2)
        cls_token = self.cls_token.expand(x.size(0), -1, -1)
        x = torch.cat((cls_token, x), dim=1)
        x = self.pos_drop(x + self.pos_embed)
        for block in self.blocks:
            x = block(x)
        x = self.norm(x)
        return self.head(x[:, 0])


def install_lens_norm(model):
    for block in model.blocks:
        block.norm1 = LENSNorm(model.dim, hidden_dim=LENS_NORM_HIDDEN)
        block.norm2 = LENSNorm(model.dim, hidden_dim=LENS_NORM_HIDDEN)
    model.norm = LENSNorm(model.dim, hidden_dim=LENS_NORM_HIDDEN)


def install_lens_max(model):
    for block in model.blocks:
        block.attn.lens_max = LENSMax(
            num_tokens=model.num_tokens,
            row_hidden=LENS_MAX_ROW_HIDDEN,
            element_hidden=LENS_MAX_ELEMENT_HIDDEN,
        )


def checkpoint_model_state(checkpoint):
    return checkpoint["model"] if "model" in checkpoint else checkpoint


def experiment_config():
    return {
        "experiment": EXPERIMENT_NAME,
        "baseline_family": BASELINE_FAMILY,
        "baseline_norm": BASELINE_NORM,
        "baseline_attention": "softmax",
        "rms_norm_eps": RMS_NORM_EPS,
        "dyt_alpha_init": DYT_ALPHA_INIT,
        "from_pretrained": FROM_PRETRAINED,
        "use_lens_norm": USE_LENS_NORM,
        "use_lens_max": USE_LENS_MAX,
        "seeds": SEEDS,
        "device": str(DEVICE),
        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "batch_size": BATCH_SIZE,
        "weight_decay": WEIGHT_DECAY,
        "grad_clip_norm": GRAD_CLIP_NORM,
        "image_size": IMAGE_SIZE,
        "patch_size": PATCH_SIZE,
        "embed_dim": EMBED_DIM,
        "depth": DEPTH,
        "mlp_ratio": MLP_RATIO,
        "dropout": DROPOUT,
        "attention_dropout": ATTENTION_DROPOUT,
        "validation_size": VALIDATION_SIZE,
        "lens_norm_hidden": LENS_NORM_HIDDEN,
        "lens_max_row_hidden": LENS_MAX_ROW_HIDDEN,
        "lens_max_element_hidden": LENS_MAX_ELEMENT_HIDDEN,
    }


def build_experiment_model(seed):
    model = CIFAR100SHViT(
        image_size=IMAGE_SIZE,
        patch_size=PATCH_SIZE,
        num_classes=NUM_CLASSES,
        dim=EMBED_DIM,
        depth=DEPTH,
        mlp_ratio=MLP_RATIO,
        drop=DROPOUT,
        attn_drop=ATTENTION_DROPOUT,
    )

    if FROM_PRETRAINED:
        baseline_path = RUNS_ROOT / f"{BASELINE_EXPERIMENT_NAME}_seed{seed}" / "best.pt"
        if not baseline_path.exists():
            raise FileNotFoundError(f"Missing baseline checkpoint: {baseline_path}")
        checkpoint = torch.load(baseline_path, map_location="cpu")
        baseline_config = checkpoint.get("experiment_config", {})
        if (baseline_config.get("experiment") != BASELINE_EXPERIMENT_NAME
                or baseline_config.get("baseline_family") != BASELINE_FAMILY
                or baseline_config.get("baseline_attention") != "softmax"
                or baseline_config.get("use_lens_norm") is not False
                or baseline_config.get("use_lens_max") is not False):
            raise ValueError(f"Checkpoint is not the matching family baseline: {baseline_path}")
        model.load_state_dict(checkpoint_model_state(checkpoint), strict=True)
        print(f"Loaded baseline: {baseline_path}")

    if USE_LENS_NORM:
        install_lens_norm(model)
    if USE_LENS_MAX:
        install_lens_max(model)

    return model


In [4]:
CIFAR100_MEAN = (0.5071, 0.4867, 0.4408)
CIFAR100_STD = (0.2675, 0.2565, 0.2761)


train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])


def make_loaders(seed):
    train_dataset = datasets.CIFAR100(
        root=DATA_ROOT,
        train=True,
        download=True,
        transform=train_transform,
    )
    validation_dataset = datasets.CIFAR100(
        root=DATA_ROOT,
        train=True,
        download=False,
        transform=eval_transform,
    )
    test_dataset = datasets.CIFAR100(
        root=DATA_ROOT,
        train=False,
        download=True,
        transform=eval_transform,
    )

    split_generator = torch.Generator().manual_seed(seed)
    indices = torch.randperm(len(train_dataset), generator=split_generator).tolist()
    validation_indices = indices[:VALIDATION_SIZE]
    training_indices = indices[VALIDATION_SIZE:]

    training_subset = Subset(train_dataset, training_indices)
    validation_subset = Subset(validation_dataset, validation_indices)
    loader_generator = torch.Generator().manual_seed(seed)
    pin_memory = DEVICE.type == "cuda"

    train_loader = DataLoader(
        training_subset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=pin_memory,
        persistent_workers=NUM_WORKERS > 0,
        generator=loader_generator,
    )
    validation_loader = DataLoader(
        validation_subset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=pin_memory,
        persistent_workers=NUM_WORKERS > 0,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=pin_memory,
        persistent_workers=NUM_WORKERS > 0,
    )
    return train_loader, validation_loader, test_loader


In [5]:
@torch.no_grad()
def evaluate(model, data_loader):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for images, targets in data_loader:
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)
        logits = model(images)
        loss = F.cross_entropy(logits, targets)
        total_loss += loss.item() * targets.size(0)
        total_correct += (logits.argmax(dim=1) == targets).sum().item()
        total_examples += targets.size(0)

    return {
        "loss": total_loss / total_examples,
        "accuracy": total_correct / total_examples,
    }


def optimizer_to_device(optimizer):
    for state in optimizer.state.values():
        for key, value in state.items():
            if torch.is_tensor(value):
                state[key] = value.to(DEVICE)


def save_json(path, value):
    path.write_text(json.dumps(value, indent=2), encoding="utf-8")


def train_model(model, train_loader, validation_loader, output_dir):
    output_dir.mkdir(parents=True, exist_ok=True)
    save_json(output_dir / "config.json", experiment_config())
    best_path = output_dir / "best.pt"
    last_path = output_dir / "last.pt"
    history_path = output_dir / "history.json"
    log_path = output_dir / "train.log"

    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    start_epoch = 1
    best_validation_accuracy = -1.0
    history = []

    if last_path.exists():
        checkpoint = torch.load(last_path, map_location="cpu")
        if checkpoint.get("experiment_config") != experiment_config():
            raise ValueError(f"Resume configuration mismatch: {last_path}. Use a fresh output directory.")
        model.load_state_dict(checkpoint_model_state(checkpoint), strict=True)
        optimizer.load_state_dict(checkpoint["optimizer"])
        optimizer_to_device(optimizer)
        scheduler.load_state_dict(checkpoint["scheduler"])
        start_epoch = checkpoint["epoch"] + 1
        best_validation_accuracy = checkpoint["best_validation_accuracy"]
        history = checkpoint.get("history", [])

    for epoch in range(start_epoch, EPOCHS + 1):
        model.train()
        epoch_start = time.time()
        total_loss = 0.0
        total_correct = 0
        total_examples = 0

        for images, targets in train_loader:
            images = images.to(DEVICE, non_blocking=True)
            targets = targets.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = F.cross_entropy(logits, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            optimizer.step()

            total_loss += loss.item() * targets.size(0)
            total_correct += (logits.argmax(dim=1) == targets).sum().item()
            total_examples += targets.size(0)

        validation_metrics = evaluate(model, validation_loader)
        learning_rate = optimizer.param_groups[0]["lr"]
        scheduler.step()

        epoch_metrics = {
            "epoch": epoch,
            "learning_rate": learning_rate,
            "train_loss": total_loss / total_examples,
            "train_accuracy": total_correct / total_examples,
            "validation_loss": validation_metrics["loss"],
            "validation_accuracy": validation_metrics["accuracy"],
            "seconds": time.time() - epoch_start,
        }
        history.append(epoch_metrics)

        checkpoint = {
            "experiment_config": experiment_config(),
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "epoch": epoch,
            "best_validation_accuracy": max(
                best_validation_accuracy,
                validation_metrics["accuracy"],
            ),
            "history": history,
        }
        torch.save(checkpoint, last_path)

        if validation_metrics["accuracy"] > best_validation_accuracy:
            best_validation_accuracy = validation_metrics["accuracy"]
            checkpoint["best_validation_accuracy"] = best_validation_accuracy
            torch.save(checkpoint, best_path)

        save_json(history_path, history)
        message = (
            f"Epoch {epoch:03d}/{EPOCHS} | lr={learning_rate:.3e} | "
            f"train loss={epoch_metrics['train_loss']:.4f} "
            f"acc={epoch_metrics['train_accuracy']:.4f} | "
            f"val loss={validation_metrics['loss']:.4f} "
            f"acc={validation_metrics['accuracy']:.4f} | "
            f"time={epoch_metrics['seconds'] / 60:.1f}m"
        )
        print(message)
        with log_path.open("a", encoding="utf-8") as log_file:
            log_file.write(message + "\n")

    return best_path


In [6]:
results = []

for seed in SEEDS:
    print(f"\n=== {EXPERIMENT_NAME} | seed={seed} ===")
    seed_everything(seed)
    output_dir = RUNS_ROOT / f"{EXPERIMENT_NAME}_seed{seed}"
    train_loader, validation_loader, test_loader = make_loaders(seed)
    model = build_experiment_model(seed).to(DEVICE)

    total_parameters = sum(parameter.numel() for parameter in model.parameters())
    trainable_parameters = sum(
        parameter.numel() for parameter in model.parameters() if parameter.requires_grad
    )
    print(f"parameters={total_parameters:,} | trainable={trainable_parameters:,}")

    best_path = train_model(model, train_loader, validation_loader, output_dir)
    best_checkpoint = torch.load(best_path, map_location="cpu")
    model.load_state_dict(checkpoint_model_state(best_checkpoint), strict=True)
    model.to(DEVICE)

    validation_metrics = evaluate(model, validation_loader)
    test_metrics = evaluate(model, test_loader)
    seed_result = {
        "seed": seed,
        "experiment_config": experiment_config(),
        "run_directory": str(output_dir),
        "best_epoch": best_checkpoint["epoch"],
        "validation": validation_metrics,
        "test": test_metrics,
        "total_parameters": total_parameters,
        "trainable_parameters": trainable_parameters,
    }
    save_json(output_dir / "metrics.json", seed_result)
    results.append(seed_result)
    print(
        f"best epoch={seed_result['best_epoch']} | "
        f"val acc={validation_metrics['accuracy']:.4f} | "
        f"test acc={test_metrics['accuracy']:.4f}"
    )

    del model, train_loader, validation_loader, test_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


test_accuracies = np.array([result["test"]["accuracy"] for result in results])
summary = {
    "experiment": EXPERIMENT_NAME,
    "seeds": SEEDS,
    "results": results,
    "test_accuracy_mean": float(test_accuracies.mean()),
    "test_accuracy_std": None,  # Single seed: standard deviation is not estimated.
}
save_json(RUNS_ROOT / f"{EXPERIMENT_NAME}_summary.json", summary)
print(
    f"\n{EXPERIMENT_NAME}: "
    f"test accuracy={summary['test_accuracy_mean']:.4f} | seed=42 (single run)"
)



=== 1_cifar100_shvit_dyt_softmax_baseline | seed=42 ===


Files already downloaded and verified


Files already downloaded and verified


parameters=622,699 | trainable=622,699


Epoch 001/400 | lr=1.000e-03 | train loss=4.2955 acc=0.0407 | val loss=4.1137 acc=0.0626 | time=0.1m


Epoch 002/400 | lr=1.000e-03 | train loss=3.9226 acc=0.0872 | val loss=3.7801 acc=0.1156 | time=0.0m


... (396 outputs left out)

Epoch 399/400 | lr=6.168e-08 | train loss=0.8144 acc=0.7500 | val loss=2.4716 acc=0.4634 | time=0.0m


Epoch 400/400 | lr=1.542e-08 | train loss=0.8159 acc=0.7478 | val loss=2.4716 acc=0.4636 | time=0.0m


best epoch=323 | val acc=0.4668 | test acc=0.4584



1_cifar100_shvit_dyt_softmax_baseline: test accuracy=0.4584 | seed=42 (single run)
